### General definitions

In [1]:
class FasterAnalyzer:
    # uses Armenian Uniparser, adds caching for speed

    def __init__(self, parser):
        self.parser = parser
        self._cache = {}

    def analyze_words(self, word):
        # analyzes a word, returns analyses

        if word in self._cache:
        # check if already processed and in cache
            return self._cache[word]

        else:
            # process with analyzer
            analyses = self.parser.analyze_words(word, format='json')

            # add to cache
            self._cache[word] = analyses
            return analyses

In [2]:
def arm_remove_punct(token):
    # string.punctuation + "Cyrillic" quotation marks + Armenian full stop
    # + Armenian comma ՝ + ` (found in OCR in place of ՝)
    token = token.strip('''!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~«»։՝`''')
    # Armenian possibly word-internal punctuation symbols
    token = token.replace('՛', '')
    token = token.replace('՜', '')
    token = token.replace('՞', '')
    return token

<>:4: SyntaxWarning: invalid escape sequence '\]'
<>:4: SyntaxWarning: invalid escape sequence '\]'
/tmp/ipykernel_29653/1194923447.py:4: SyntaxWarning: invalid escape sequence '\]'
  token = token.strip('''!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~«»։՝`''')


In [3]:
# # older code that shows nice way to look at the parses
# df = pd.DataFrame(all_parses)
# df[df['wf'] == 'ինչ']

### Data

In [4]:
import pandas as pd

In [5]:
df = pd.read_pickle('data.pkl')
df

,title,text,date,source,genre,tokens_number
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93
3,ՄԱՇԻՆԸ,"ՄԱՇԻՆԸ \n\nԳասպար, Խաթուն միասին, Քաղաք էշտալ ...",1971,Atamanyan_2006,poem,98
4,ԹՈՓՏԻՆ ԵՎ ԹՈՓՑԻԿԸ,ԹՈՓՏԻՆ ԵՎ ԹՈՓՑԻԿԸ \n\nԻնչքան տարուք դուն կառնո...,1973,Atamanyan_2006,poem,153
...,...,...,...,...,...,...
163,ԵՐԳ ՔԵՖԵԻ ՄԱՍԻՆ,"ԵՐԳ ՔԵՖԵԻ ՄԱՍԻՆ\n\nՔէֆէ քաղաք, բայձառ քաղաք, \...",,Dagldiyan_2023,unknown,69
164,ՍԱՆԴՌԻ ԵՐԳԸ,"ՍԱՆԴՌԻ ԵՐԳԸ\n\nՀէնդէք ընգա, սանդըռ կըդա, \nՍան...",,Dagldiyan_2023,unknown,65
165,ԳԱՐՋՈՒԳ ՄԱՌՏԻՆ,"ԳԱՐՋՈՒԳ ՄԱՌՏԻՆ\n\nՀէ՜յ, մէգդի գացէք մէյդան, \n...",,Dagldiyan_2023,unknown,150
166,ՑԱՓԷԹ,ՑԱՓԷԹ\n\nԿըլլայի – տա ան ատէնը էռսուն-էռսունհի...,,Dagldiyan_2023,unknown,482


In [ ]:
def tokenize(text):
    from re import split
    return split(r'\s+', text)

def preprocess(text):
      t = tokenize(text)
      words = [arm_remove_punct(word).lower() for word in t]
      return words

In [ ]:
df['preprocessed'] = df['text'].apply(preprocess)
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ..."
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո..."
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,..."


In [ ]:
def get_unique_tokens(preprocessed_text):
    unique_tokens = set()
    for l in preprocessed_text:
        unique_tokens.update(l)
    return list(unique_tokens)

In [ ]:
unique_strings_list = get_unique_tokens(df['preprocessed'])
print(len(unique_strings_list))
print(unique_strings_list[:10])

22195
['', 'տեղերը', 'ագեր', 'չայնին', 'կավատ-կավատ', 'ծխոտը', 'կըբռնե', 'ապուլ', 'օշթանջա', 'էլլելեն']


### from Ira

In [6]:
# raw_1 = []
# raw_2 = []

# for index, row in parses.iterrows():
#     r_1 = row['after_spellcheck']
#     r_2 = row['after_after_spellcheck']
#     if not pd.isna(r_1):
#         r1 = r_1.split("'wf'")
#         r1 = r1[1:]
#         for r in r1:
#             raw_1.append(r)
#     if not pd.isna(r_2):
#         r2 = r_2.split("'wf'")
#         r2 = r2[1:]
#         for r in r2:
#             raw_2.append(r)

# raw_1 = [raw.split("'")[1] for raw in raw_1]
# raw_2 = [raw.split("'")[1] for raw in raw_2]

NameError: name 'parses' is not defined

In [ ]:
# counted_1 = []
# counted_2 = []

# analyses = a.analyze_words(raw_1, format='json')
# for analysis in analyses:
#     if (analysis[0]['lemma']) != '':
#         counted_1.append(len(analysis))
#     else:
#         counted_1.append(0)

# counted_1_2 = [c for c in counted_1 if c!=0]

# analyses = a.analyze_words(raw_2, format='json')
# for analysis in analyses:
#     if (analysis[0]['lemma']) != '':
#         counted_2.append(len(analysis))
#     else:
#         counted_2.append(0)

# counted_2_2 = [c for c in counted_2 if c!=0]

### Import parsers

Eastern Armenian Uniparser

In [7]:
!pip install uniparser-eastern-armenian

In [8]:
from uniparser_eastern_armenian import EasternArmenianAnalyzer

In [9]:
analyzer_EA = FasterAnalyzer(EasternArmenianAnalyzer())
analyzer_EA

In [10]:
analyzer_EA.analyze_words('մարդ')

[{'wf': 'մարդ',
  'lemma': 'մարդ',
  'gramm': ['N', 'anim', 'hum', 'sg', 'nom', 'nonposs'],
  'wfGlossed': 'մարդ',
  'gloss': 'person',
  'trans_en': 'man, person, soul'}]

Western Armenian Uniparser

In [11]:
!pip install uniparser_morph

In [12]:
from uniparser_morph import Analyzer

Manually upload files archive

In [13]:
!unzip 'western_uniparser' -d './western'

Archive:  western_uniparser.zip
replace ./western/lexemes/hyw-lexemes.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./western/lexemes/hyw-lexemes_irregular.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./western/lexemes/hyw-lexemes_PN_wikt.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./western/lexemes/hyw-lexemes_PRO.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./western/paradigms/hyw-paradigms_N.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./western/paradigms/hyw-paradigms_rest.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./western/paradigms/hyw-paradigms_V.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [14]:
analyzer_WA = Analyzer()
analyzer_WA.lexFile = './western/lexemes/'
analyzer_WA.paradigmFile = './western/paradigms/'
analyzer_WA.load_grammar()

analyzer_WA = FasterAnalyzer(analyzer_WA)

In [15]:
analyzer_WA.analyze_words('մարդ')

[{'wf': 'մարդ',
  'lemma': 'մարդ',
  'gramm': ['N', 'hum', 'anim', 'sg', 'nom'],
  'wfGlossed': 'մարդ',
  'gloss': 'STEM',
  'trans_en': 'man, person',
  'trans_fr': 'homme, humain'},
 {'wf': 'մարդ',
  'lemma': 'մար',
  'gramm': ['N', 'inanim', 'sg', 'nom', 'poss.2'],
  'wfGlossed': 'մար-դ',
  'gloss': 'STEM-2POSS',
  'trans_en': 'width, breath of linen; amphora',
  'trans_fr': 'mesure équivalant à une quarantaine de litres'},
 {'wf': 'մարդ',
  'lemma': 'մար',
  'gramm': ['N', 'inanim', 'sg', 'nom', 'poss.2'],
  'wfGlossed': 'մար-դ',
  'gloss': 'STEM-2POSS',
  'trans_en': '',
  'trans_fr': 'mède'}]

Customized Nor_Nakhichevan Armenian Uniparser

In [16]:
# !pip install uniparser_morph
# from uniparser_morph import Analyzer

In [17]:
!git clone 'https://github.com/NorNakhichevan-Armenian-corpus/uniparser-grammar-nornakhichevan-armenian.git'

fatal: destination path 'uniparser-grammar-nornakhichevan-armenian' already exists and is not an empty directory.


In [18]:
analyzer_NNA = Analyzer()
analyzer_NNA.lexFile = './uniparser-grammar-nornakhichevan-armenian/lexemes/'
analyzer_NNA.paradigmFile = './uniparser-grammar-nornakhichevan-armenian/paradigms/'
analyzer_NNA.load_grammar()

analyzer_NNA = FasterAnalyzer(analyzer_NNA)

In [19]:
analyzer_NNA.analyze_words('մարդ')

[{'wf': 'մարդ',
  'lemma': 'մարդ',
  'gramm': ['N', 'anim', 'hum', 'sg', 'nom', 'nonposs'],
  'wfGlossed': 'մարդ',
  'gloss': 'person',
  'trans_en': 'man, person, soul',
  'trans_ru': ''},
 {'wf': 'մարդ',
  'lemma': 'մար',
  'gramm': ['N', 'sg', 'nom', 'poss.2'],
  'wfGlossed': 'մար-դ',
  'gloss': 'STEM-2POSS',
  'trans_en': '',
  'trans_ru': 'мать'}]

### More definitions for benchmarking

In [27]:
def make_analyses(preprocessed_text, analyzer):
    return [analyzer.analyze_words(w) for w in preprocessed_text]

In [39]:
def count_parses(parses_list):
    # counts all cases when there is an analysis for all tokens in prepared list
    ct = 0
    for token_analyses in parses_list:
        for analysis in token_analyses:
            if analysis['lemma'] != '':
                ct += 1
    return ct

In [42]:
def count_parses_total(parses_list):
    # counts the total number of analyses for all tokens in prepared list
    ct = 0
    for token_analyses in parses_list:
        for analysis in token_analyses:
            if analysis['lemma'] != '':
                # write down how many analyses there were
                ct += len(token_analyses)
    return ct

### Eastern Armenian Uniparser

In [29]:
df['analyses_EA'] = df['preprocessed'].apply(make_analyses, args=(analyzer_EA,))
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ..."
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR..."
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR..."


In [40]:
df['count_EA'] = df['analyses_EA'].apply(count_parses)
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA,count_WA,analyses_NNA,count_NNA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",53,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",66
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",57,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",139,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",127
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",136,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",102


In [43]:
df['total_count_EA'] = df['analyses_EA'].apply(count_parses_total)
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA,count_WA,analyses_NNA,count_NNA,total_count_EA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",53,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",66,54
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",57,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",139,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",127,91
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",136,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",102,96


In [66]:
print(sum(df['count_EA']),
      round(sum(df['count_EA']) / sum(df['tokens_number']), 2))
print(sum(df['total_count_EA']),
      round(sum(df['total_count_EA']) / sum(df['tokens_number']), 2))

69511 0.68
132329 1.29


In [56]:
unique_parses_EA = make_analyses(unique_strings_list, analyzer_EA)
print(len(unique_parses_EA))
print(unique_parses_EA[1])

22195
[{'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['N', 'inanim', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'place-PL-DEF', 'trans_en': 'place, locality, location'}, {'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['A', 'oral', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'STEM-PL-DEF', 'trans_en': ''}, {'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['PRON', 'ADV', 'oral', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'there-PL-DEF', 'trans_en': 'there'}]


In [67]:
print(count_parses(unique_parses_EA),
      round(count_parses(unique_parses_EA) / len(unique_parses_EA), 2))
print(count_parses_total(unique_parses_EA),
      round(count_parses_total(unique_parses_EA) / len(unique_parses_EA), 2))

9026 0.41
16248 0.73


### Wetern Armenian Uniparser

In [31]:
df['analyses_WA'] = df['preprocessed'].apply(make_analyses, args=(analyzer_WA,))
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",54,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ..."
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",91,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR..."
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",96,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR..."


In [46]:
df['count_WA'] = df['analyses_WA'].apply(count_parses)
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA,count_WA,analyses_NNA,count_NNA,total_count_EA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",39,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",66,54
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",57,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",61,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",127,91
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",72,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",102,96


In [47]:
df['total_count_WA'] = df['analyses_WA'].apply(count_parses_total)
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA,count_WA,analyses_NNA,count_NNA,total_count_EA,total_count_WA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",39,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",66,54,53
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",57,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",61,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",127,91,139
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",72,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",102,96,136


In [68]:
print(sum(df['count_WA']),
      round(sum(df['count_WA']) / sum(df['tokens_number']), 2))
print(sum(df['total_count_WA']),
      round(sum(df['total_count_WA']) / sum(df['tokens_number']), 2))

70765 0.69
136411 1.33


In [59]:
unique_parses_WA = make_analyses(unique_strings_list, analyzer_WA)
print(len(unique_parses_WA))
print(unique_parses_WA[1])

22195
[{'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['N', 'inanim', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'STEM-DEF', 'trans_en': 'place', 'trans_fr': 'lieu'}, {'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['POST', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'STEM-DEF', 'trans_en': 'place', 'trans_fr': ''}]


In [69]:
print(count_parses(unique_parses_WA),
      round(count_parses(unique_parses_WA) / len(unique_parses_WA), 2))
print(count_parses_total(unique_parses_WA),
      round(count_parses_total(unique_parses_WA) / len(unique_parses_WA), 2))

8352 0.38
15630 0.7


### Nor-Nakhichevan Armenian Uniparser

In [33]:
df['analyses_NNA'] = df['preprocessed'].apply(make_analyses, args=(analyzer_NNA,))
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA,count_WA,analyses_NNA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",54,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",53,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ..."
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",91,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",139,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR..."
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",96,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",136,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR..."


In [49]:
df['count_NNA'] = df['analyses_NNA'].apply(count_parses)
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA,count_WA,analyses_NNA,count_NNA,total_count_EA,total_count_WA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",39,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,54,53
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",57,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",61,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",67,91,139
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",72,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,96,136


In [50]:
df['total_count_NNA'] = df['analyses_NNA'].apply(count_parses_total)
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA,count_WA,analyses_NNA,count_NNA,total_count_EA,total_count_WA,total_count_NNA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",39,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,54,53,66
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",57,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",61,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",67,91,139,127
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",72,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,96,136,102


In [70]:
print(sum(df['count_NNA']),
      round(sum(df['count_NNA']) / sum(df['tokens_number']), 2))
print(sum(df['total_count_NNA']),
      round(sum(df['total_count_NNA']) / sum(df['tokens_number']), 2))

91534 0.89
244916 2.39


In [61]:
unique_parses_NNA = make_analyses(unique_strings_list, analyzer_NNA)
print(len(unique_parses_NNA))
print(unique_parses_NNA[1])

22195
[{'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['N', 'inanim', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'place-PL-DEF', 'trans_en': 'place, locality, location', 'trans_ru': 'место'}, {'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['PRON', 'ADV', 'oral', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'there-PL-DEF', 'trans_en': 'there', 'trans_ru': ''}, {'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['A', 'oral', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'STEM-PL-DEF', 'trans_en': '', 'trans_ru': ''}]


In [71]:
print(count_parses(unique_parses_NNA),
      round(count_parses(unique_parses_NNA) / len(unique_parses_NNA), 2))
print(count_parses_total(unique_parses_NNA),
      round(count_parses_total(unique_parses_NNA) / len(unique_parses_NNA), 2))

8113 0.37
12463 0.56


In [74]:
df.to_pickle('data2.pkl')